In [12]:
#pip install ortools

In [ ]:
#pip install holidayskr

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [holidayskr]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# pip install ortools
from ortools.sat.python import cp_model
from datetime import date, timedelta

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "김영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명(규칙 대상)
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 2

# "전체 기준" 일일 필요 인원
REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# "전체 기준" N 필요인원(1~2) 일자별
# 비워두면 평일2/주말1 자동 생성
N_REQ_TOTAL = []

# (선택) 공휴일 index(0-based)
HOLIDAY_INDEXES = []

# =========================================================
# 2) 달력 유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    cur = date(year, month, 1)
    return (nxt - cur).days

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)

# 기본 N 요구 생성
if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)  # 기본: 평일2 / 주말1
    N_REQ_TOTAL = tmp

assert len(N_REQ_TOTAL) == NUM_DAYS
assert all(n in [1, 2] for n in N_REQ_TOTAL)

# =========================================================
# 3) 모델 구성 (A도 변수로 포함, 단 N 금지)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)        # 9명
P_NONA = len(STAFF_IDS)     # 8명

idxA = 0
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
days = range(NUM_DAYS)
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

# x[i,d,s] ∈ {0,1}  (i는 ALL_IDS 기준: 0=A, 1~8=비A)
x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개(근무/휴무 포함)
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# -------------------------
# A(김영철)는 N 금지 (요청 반영)
# -------------------------
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

# -------------------------
# 일일 인원 충족 (전체 기준: A 포함)
# -------------------------
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 4) 하드 규칙 (비A 8명에게만 적용되는 것)
# =========================================================
# 비A 인덱스: 1~8
people_nonA = range(1, P_ALL)

# (하드) N 정확히 6회 (비A만)
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# (하드) N 최대 2연속 (비A만)
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d + 1, SHIFT_N] + x[i, d + 2, SHIFT_N] <= 2)

# (하드) N 다음날 D/E 금지(OFF 또는 N만) (비A만)
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d + 1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d + 1, SHIFT_E] <= 1)

# (하드) N 2일 후 D 금지 (비A만)
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d + 2, SHIFT_D] <= 1)

# (하드) E 다음날 D 금지 (비A만)
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d + 1, SHIFT_D] <= 1)

# (하드) B/C 특별 조건 (전체 N 기준에 따라)
for d in days:
    if N_REQ_TOTAL[d] == 1:
        # N이 1명인 날: B/C는 N 금지(혼자 N 불가)
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        # N이 2명인 날: B와 C 동시 N 금지
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 5) 소프트 규칙(가능하면 맞추기) - 비A 기준
# =========================================================
D_cnt, E_cnt, OFF_cnt, WORK_cnt = {}, {}, {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    OFF_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{ALL_IDS[i]}")
    WORK_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"WORKcnt_{ALL_IDS[i]}")

    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))
    model.Add(OFF_cnt[i] == sum(x[i, d, SHIFT_OFF] for d in days))
    model.Add(WORK_cnt[i] == sum(x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N] for d in days))

# 개인 D=E 목표(소프트)
DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

# 전원 D/E 편차 최소화(소프트)
maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# 휴무/근무 목표(소프트, 비A만)
OFF_target = 11
WORK_target = 17
OFF_dev, WORK_dev = {}, {}
for i in people_nonA:
    OFF_dev[i] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{ALL_IDS[i]}")
    WORK_dev[i] = model.NewIntVar(0, NUM_DAYS, f"WORKdev_{ALL_IDS[i]}")

    off_diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{ALL_IDS[i]}")
    work_diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"WORKdiff_{ALL_IDS[i]}")
    model.Add(off_diff == OFF_cnt[i] - OFF_target)
    model.Add(work_diff == WORK_cnt[i] - WORK_target)
    model.AddAbsEquality(OFF_dev[i], off_diff)
    model.AddAbsEquality(WORK_dev[i], work_diff)

# 휴무 3연속 지양(소프트, 비A만)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d + 1, SHIFT_OFF] + x[i, d + 2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d + 1, SHIFT_OFF] + x[i, d + 2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# 일요일 D 월 1회 이상(소프트, 비A만)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# 주말/휴일 편중 최소화(소프트, 비A만)
special = set(HOLIDAY_INDEXES)
for d in days:
    if is_weekend(START_DATE + timedelta(days=d)):
        special.add(d)
special = sorted(list(special))

special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special
    ))

maxSP = model.NewIntVar(0, len(special), "maxSP")
minSP = model.NewIntVar(0, len(special), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 6) 목적함수(페널티)
# =========================================================
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500
W_DE_GAP = 30
W_FAIR_D = 20
W_FAIR_E = 20
W_OFF_TARGET = 10
W_WORK_TARGET = 10
W_SPECIAL_FAIR = 15

model.Minimize(
    W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_OFF_TARGET * sum(OFF_dev.values())
    + W_WORK_TARGET * sum(WORK_dev.values())
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 7) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 8) 출력: 이름(왼쪽 행) / 날짜(오른쪽 열)
# =========================================================
date_headers = []
for d in range(NUM_DAYS):
    dt = START_DATE + timedelta(days=d)
    date_headers.append(f"{dt.day:02d}({weekday_kor(dt)})")

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "일D"]))

def get_row_shifts(i: int):
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row_shifts = get_row_shifts(i)

    Dn = row_shifts.count("D")
    En = row_shifts.count("E")
    Nn = row_shifts.count("N")
    Wk = Dn + En + Nn
    Of = row_shifts.count("-")
    sunD = sum(1 for d in sunday_indexes if row_shifts[d] == "D")

    print("\t".join([name] + row_shifts + [str(Dn), str(En), str(Nn), str(Wk), str(Of), str(sunD)]))

# =========================================================
# 9) (선택) 날짜별 인원 체크(전체 기준)
# =========================================================
print("\n== 날짜별 인원 체크(전체 기준) ==")
print("\t".join(["항목"] + date_headers))

D_line = ["D(전체)"]
E_line = ["E(전체)"]
N_line = ["N(전체)"]
OFF_line = ["-(전체)"]

for d in range(NUM_DAYS):
    d_cnt = sum(solver.Value(x[i, d, SHIFT_D]) for i in people_all)
    e_cnt = sum(solver.Value(x[i, d, SHIFT_E]) for i in people_all)
    n_cnt = sum(solver.Value(x[i, d, SHIFT_N]) for i in people_all)
    off_cnt = sum(solver.Value(x[i, d, SHIFT_OFF]) for i in people_all)

    D_line.append(str(d_cnt))
    E_line.append(str(e_cnt))
    N_line.append(str(n_cnt))
    OFF_line.append(str(off_cnt))

print("\t".join(D_line))
print("\t".join(E_line))
print("\t".join(N_line))
print("\t".join(OFF_line))



== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	D	E	N	근무	휴무	일D
김영철	-	D	E	E	E	E	-	-	E	E	D	-	E	-	-	D	D	D	D	-	-	-	E	-	-	D	D	-	8	8	0	16	12	0
홍진우	-	N	-	-	D	D	E	E	N	-	E	-	D	E	-	N	-	N	-	-	D	D	D	E	E	N	N	-	6	6	6	18	10	1
김다영	-	-	D	D	E	E	E	-	D	N	-	E	-	D	D	-	N	-	N	N	-	E	E	N	N	-	-	D	6	6	6	18	10	1
강승민	E	E	-	E	-	N	-	-	D	N	N	-	E	-	D	N	-	-	D	D	E	E	N	N	-	-	D	D	6	6	6	18	10	1
문승환	E	-	N	N	-	-	D	D	N	-	-	E	N	-	-	D	D	N	N	-	E	-	D	D	E	E	-	E	6	6	6	18	10	1
라영일	D	E	-	D	N	N	-	E	E	-	-	D	D	D	N	-	-	D	E	E	-	-	N	-	N	-	E	N	6	6	6	18	10	1
이병욱	-	N	N	-	-	D	D	D	-	D	E	N	-	E	E	-	N	-	E	E	N	-	-	D	D	E	N	-	6	6	6	18	10	1
김동명	N	-	E	N	N	-	N	-	-	D	D	D	N	-	E	E	E	E	-	D	D	D	-	E	-	N	-	-	6	6	6	18	10	1
김선우	D	D	D	-	D	-	-	N	-	E	N	N	-	N	-	E	E	E	-	N	-	N	-	-	D	D	E	E	6	6	6	18	10	1

== 날짜별 인원 체크(전체 기준) ==
항목	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(

In [ ]:
# pip install ortools
from ortools.sat.python import cp_model
from datetime import date, timedelta

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "김영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명(규칙 대상)
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 2

REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# "전체 기준" N 필요인원(1~2) 일자별 (비워두면 평일2/주말1)
N_REQ_TOTAL = []

HOLIDAY_INDEXES = []

# =========================================================
# 2) 달력 유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    cur = date(year, month, 1)
    return (nxt - cur).days

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)
    N_REQ_TOTAL = tmp

assert len(N_REQ_TOTAL) == NUM_DAYS
assert all(n in [1, 2] for n in N_REQ_TOTAL)

# =========================================================
# 3) 모델 구성 (A도 포함, 단 N 금지)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)
idxA = 0
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
people_nonA = range(1, P_ALL)   # 1~8
days = range(NUM_DAYS)
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# A는 N 금지
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

# 일일 인원(전체 기준)
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 4) 하드 규칙(필수)
# =========================================================
# 비A: N=6
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# 비A: N 최대 2연속
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

# 비A: N 다음날 D/E 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

# 비A: N 2일 후 D 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

# ✅ 전원: E 다음날 D 금지
for i in people_all:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

# B/C 특별조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 5) 휴무 11 하드(비A) + 연속근무 제한(전원)
# =========================================================
work = {(i, d): model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}") for i in people_all for d in days}
for i in people_all:
    for d in days:
        model.Add(work[i, d] == 1 - x[i, d, SHIFT_OFF])

# 비A: 휴무 11 정확히
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_OFF] for d in days) == 11)

# 전원: 6연속 근무 금지(=최대 5연속)
for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i, d+k] for k in range(6)) <= 5)

# 5연속 근무는 지양(소프트)
five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i, d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i, d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# =========================================================
# 6) D/E 균일화(강화된 소프트)
# =========================================================
# 비A 기준으로 D/E 카운트
D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

# 개인별 |D - E| 최소화
DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

# 사람간 D/E 편차 최소화
maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# =========================================================
# 7) 기타 소프트(이전 코드 그대로)
# =========================================================
# 휴무 3연속 지양(비A)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# 일요일 D 1회 이상(비A, 소프트)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# 주말/휴일 편차(비A, 소프트)
special = set(HOLIDAY_INDEXES)
for d in days:
    if is_weekend(START_DATE + timedelta(days=d)):
        special.add(d)
special = sorted(list(special))

special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special
    ))

maxSP = model.NewIntVar(0, len(special), "maxSP")
minSP = model.NewIntVar(0, len(special), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 8) 목적함수(페널티) - ✅ D/E 균일화 가중치 강화
# =========================================================
W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500

# ✅ 여기 3개를 크게 올림(균일화 강제 느낌)
W_DE_GAP = 200      # 개인 D와 E 차이 최소화
W_FAIR_D = 150      # 사람간 D 편차
W_FAIR_E = 150      # 사람간 E 편차

W_SPECIAL_FAIR = 15

model.Minimize(
    W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 9) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 10) 출력: 이름(왼쪽 행) / 날짜(오른쪽 열)
# =========================================================
date_headers = []
for d in range(NUM_DAYS):
    dt = START_DATE + timedelta(days=d)
    date_headers.append(f"{dt.day:02d}({weekday_kor(dt)})")

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "일D", "5연속", "|D-E|"]))

def get_row_shifts(i: int):
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row_shifts = get_row_shifts(i)

    Dn = row_shifts.count("D")
    En = row_shifts.count("E")
    Nn = row_shifts.count("N")
    Wk = Dn + En + Nn
    Of = row_shifts.count("-")
    sunD = sum(1 for d in sunday_indexes if row_shifts[d] == "D")

    five_cnt = 0
    for d0 in range(NUM_DAYS - 4):
        if all(row_shifts[d0 + k] != "-" for k in range(5)):
            five_cnt += 1

    print("\t".join([
        name, *row_shifts,
        str(Dn), str(En), str(Nn), str(Wk), str(Of), str(sunD),
        str(five_cnt), str(abs(Dn - En))
    ]))

print("\n== 비A D/E 편차 체크 ==")
print("D 편차(max-min):", solver.Value(maxD) - solver.Value(minD))
print("E 편차(max-min):", solver.Value(maxE) - solver.Value(minE))

# =========================================================
# 11) 날짜별 인원 체크(전체 기준)
# =========================================================
print("\n== 날짜별 인원 체크(전체 기준) ==")
print("\t".join(["항목"] + date_headers))

D_line = ["D(전체)"]
E_line = ["E(전체)"]
N_line = ["N(전체)"]
OFF_line = ["-(전체)"]

for d in range(NUM_DAYS):
    d_cnt = sum(solver.Value(x[i, d, SHIFT_D]) for i in people_all)
    e_cnt = sum(solver.Value(x[i, d, SHIFT_E]) for i in people_all)
    n_cnt = sum(solver.Value(x[i, d, SHIFT_N]) for i in people_all)
    off_cnt = sum(solver.Value(x[i, d, SHIFT_OFF]) for i in people_all)

    D_line.append(str(d_cnt))
    E_line.append(str(e_cnt))
    N_line.append(str(n_cnt))
    OFF_line.append(str(off_cnt))

print("\t".join(D_line))
print("\t".join(E_line))
print("\t".join(N_line))
print("\t".join(OFF_line))



== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	D	E	N	근무	휴무	일D	5연속	|D-E|
김영철	E	E	E	E	E	-	E	E	E	E	E	-	D	E	E	E	E	-	D	D	D	E	E	-	D	D	D	D	8	16	0	24	4	0	4	8
홍진우	-	N	N	-	N	-	-	D	D	E	N	-	E	E	E	-	D	N	N	-	E	-	-	D	D	D	-	-	6	5	6	17	11	1	0	1
김다영	E	E	-	-	D	N	-	-	D	N	-	-	D	D	-	N	N	-	E	-	D	D	-	E	E	N	N	-	6	5	6	17	11	1	0	1
강승민	D	-	-	D	E	E	E	-	-	D	D	D	-	D	-	E	N	-	E	N	-	-	N	N	-	N	-	N	6	5	6	17	11	1	0	1
문승환	-	-	D	N	N	-	-	E	N	-	N	N	-	-	D	D	-	D	-	N	-	E	E	E	E	-	D	D	6	5	6	17	11	1	0	1
라영일	-	D	N	N	-	E	N	-	-	D	D	D	-	-	D	D	E	E	-	E	N	N	-	N	-	E	-	-	6	5	6	17	11	1	0	1
이병욱	N	N	-	E	-	D	D	D	-	N	-	N	N	-	-	N	-	E	-	D	-	-	D	D	-	E	E	E	6	5	6	17	11	1	0	1
김동명	D	D	E	-	D	N	-	N	N	-	-	E	E	-	N	-	-	D	D	E	E	-	D	-	N	-	N	-	6	5	6	17	11	1	0	1
김선우	-	-	D	D	-	D	D	-	E	-	E	E	N	N	-	-	D	N	N	-	-	D	N	-	N	-	E	E	6	5	6	17	11	1	0	1

== 비A D/E 편차 체크 ==
D 편차(max-min): 0
E 편차(max-min): 0

== 날짜별 인원 체크(전체 기준)

In [ ]:
# pip install ortools
from ortools.sat.python import cp_model
from datetime import date, timedelta

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "김영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명(규칙 대상)
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 2

REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# "전체 기준" N 필요인원(1~2) 일자별 (비워두면 평일2/주말1)
N_REQ_TOTAL = []

HOLIDAY_INDEXES = []

# =========================================================
# 2) 달력 유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    cur = date(year, month, 1)
    return (nxt - cur).days

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)
    N_REQ_TOTAL = tmp

assert len(N_REQ_TOTAL) == NUM_DAYS
assert all(n in [1, 2] for n in N_REQ_TOTAL)

# =========================================================
# 3) 모델 구성 (A 포함, 단 N 금지)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)
idxA = 0
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
people_nonA = range(1, P_ALL)   # 1~8
days = range(NUM_DAYS)
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# -------------------------
# A(김영철): N 금지 (하드)
# -------------------------
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

# -------------------------
# ✅ A(김영철): 휴무 11개 정확히 (하드)
# -------------------------
model.Add(sum(x[idxA, d, SHIFT_OFF] for d in days) == 11)

# -------------------------
# ✅ A(김영철): "데이 다음날은 무조건 이브닝" (하드)
#    동치로 강제: D(d) == E(d+1)
# -------------------------
for d in range(NUM_DAYS - 1):
    model.Add(x[idxA, d, SHIFT_D] == x[idxA, d + 1, SHIFT_E])

# -------------------------
# 일일 인원(전체 기준)
# -------------------------
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 4) 하드 규칙(필수) - 비A 중심
# =========================================================
# 비A: N=6
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# 비A: N 최대 2연속
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

# 비A: N 다음날 D/E 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

# 비A: N 2일 후 D 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

# 전원: E 다음날 D 금지
for i in people_all:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

# B/C 특별조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 5) 휴무 11 하드(비A) + 연속근무 제한(전원)
# =========================================================
# 비A: 휴무 11 정확히
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_OFF] for d in days) == 11)

# work[i,d] = 1 if working (D/E/N), 0 if off
work = {(i, d): model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}") for i in people_all for d in days}
for i in people_all:
    for d in days:
        model.Add(work[i, d] == 1 - x[i, d, SHIFT_OFF])

# 전원: 6연속 근무 금지(=최대 5연속)
for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i, d+k] for k in range(6)) <= 5)

# 5연속 근무는 지양(소프트)
five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i, d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i, d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# =========================================================
# 6) D/E 균일화(소프트 강화) - 비A 기준
# =========================================================
D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# =========================================================
# 7) 기타 소프트
# =========================================================
# 휴무 3연속 지양(비A)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# 일요일 D 1회 이상(비A, 소프트)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# 주말/휴일 편차(비A, 소프트)
special = set(HOLIDAY_INDEXES)
for d in days:
    if is_weekend(START_DATE + timedelta(days=d)):
        special.add(d)
special = sorted(list(special))

special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special
    ))

maxSP = model.NewIntVar(0, len(special), "maxSP")
minSP = model.NewIntVar(0, len(special), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 8) 목적함수(페널티)
# =========================================================
W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500
W_DE_GAP = 200
W_FAIR_D = 150
W_FAIR_E = 150
W_SPECIAL_FAIR = 15

model.Minimize(
    W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 9) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 10) 출력: 이름(왼쪽 행) / 날짜(오른쪽 열)
# =========================================================
date_headers = []
for d in range(NUM_DAYS):
    dt = START_DATE + timedelta(days=d)
    date_headers.append(f"{dt.day:02d}({weekday_kor(dt)})")

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "일D", "5연속", "|D-E|"]))

def get_row_shifts(i: int):
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row_shifts = get_row_shifts(i)

    Dn = row_shifts.count("D")
    En = row_shifts.count("E")
    Nn = row_shifts.count("N")
    Wk = Dn + En + Nn
    Of = row_shifts.count("-")
    sunD = sum(1 for d in sunday_indexes if row_shifts[d] == "D")

    five_cnt = 0
    for d0 in range(NUM_DAYS - 4):
        if all(row_shifts[d0 + k] != "-" for k in range(5)):
            five_cnt += 1

    print("\t".join([
        name, *row_shifts,
        str(Dn), str(En), str(Nn), str(Wk), str(Of), str(sunD),
        str(five_cnt), str(abs(Dn - En))
    ]))

print("\n== 체크: 김영철 규칙 ==")
A_row = get_row_shifts(idxA)
print("김영철 휴무(-) 개수:", A_row.count("-"), " (하드=11)")
viol = 0
for d in range(NUM_DAYS - 1):
    if A_row[d] == "D" and A_row[d + 1] != "E":
        viol += 1
print("김영철 D->다음날E 위반 건수:", viol)

# =========================================================
# 11) 날짜별 인원 체크(전체 기준)
# =========================================================
print("\n== 날짜별 인원 체크(전체 기준) ==")
print("\t".join(["항목"] + date_headers))

D_line = ["D(전체)"]
E_line = ["E(전체)"]
N_line = ["N(전체)"]
OFF_line = ["-(전체)"]

for d in range(NUM_DAYS):
    d_cnt = sum(solver.Value(x[i, d, SHIFT_D]) for i in people_all)
    e_cnt = sum(solver.Value(x[i, d, SHIFT_E]) for i in people_all)
    n_cnt = sum(solver.Value(x[i, d, SHIFT_N]) for i in people_all)
    off_cnt = sum(solver.Value(x[i, d, SHIFT_OFF]) for i in people_all)

    D_line.append(str(d_cnt))
    E_line.append(str(e_cnt))
    N_line.append(str(n_cnt))
    OFF_line.append(str(off_cnt))

print("\t".join(D_line))
print("\t".join(E_line))
print("\t".join(N_line))
print("\t".join(OFF_line))

해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)


SystemExit: 

In [ ]:
# pip install ortools
from ortools.sat.python import cp_model
from datetime import date, timedelta

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "김영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 2

REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# "전체 기준" N 필요인원(1~2) 일자별 (비워두면 평일2/주말1)
N_REQ_TOTAL = []

HOLIDAY_INDEXES = []

# =========================================================
# 2) 달력 유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    cur = date(year, month, 1)
    return (nxt - cur).days

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)  # 기본: 평일2 / 주말1
    N_REQ_TOTAL = tmp

assert len(N_REQ_TOTAL) == NUM_DAYS
assert all(n in [1, 2] for n in N_REQ_TOTAL)

# =========================================================
# 3) 모델 구성 (A 포함, 단 N 금지)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)
idxA = 0
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
people_nonA = range(1, P_ALL)   # 1~8
days = range(NUM_DAYS)
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# -------------------------
# A(김영철): N 금지 + D/E/-만 가능
# -------------------------
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)  # N 금지

# -------------------------
# 일일 인원(전체 기준)
# -------------------------
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 4) 하드 규칙(비A 핵심)
# =========================================================
# 비A: N=6 (하드)
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# 비A: N 최대 2연속
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

# 비A: N 다음날 D/E 금지(OFF 또는 N만)
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

# 비A: N 2일 후 D 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

# 전원: E 다음날 D 금지
for i in people_all:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

# B/C 특별조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        # N이 1명인 날: B/C는 N 금지
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        # N이 2명인 날: B와 C 동시 N 금지
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 5) 연속근무 제한(전원): 6연속 근무 금지 / 5연속 지양(소프트)
# =========================================================
work = {(i, d): model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}") for i in people_all for d in days}
for i in people_all:
    for d in days:
        model.Add(work[i, d] == 1 - x[i, d, SHIFT_OFF])

# 전원: 6연속 근무 금지 (=최대 5연속)
for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i, d+k] for k in range(6)) <= 5)

# 5연속 근무 지양(소프트)
five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i, d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i, d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# =========================================================
# 6) ✅ 오프 11개 "목표(소프트)"로 전원 적용
# =========================================================
OFF_target = 11
OFF_cnt = {}
OFF_dev = {}

for i in people_all:
    OFF_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{ALL_IDS[i]}")
    model.Add(OFF_cnt[i] == sum(x[i, d, SHIFT_OFF] for d in days))

    # |OFF_cnt - 11| 최소화
    OFF_dev[i] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{ALL_IDS[i]}")
    off_diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{ALL_IDS[i]}")
    model.Add(off_diff == OFF_cnt[i] - OFF_target)
    model.AddAbsEquality(OFF_dev[i], off_diff)

# =========================================================
# 7) D/E 균일화(비A 소프트 강화)
# =========================================================
D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# =========================================================
# 8) ✅ 김영철(A) 패턴 선호(소프트): D→E 선호, E→OFF 선호
# =========================================================
A_bad_DE = []   # A가 D인데 다음날 E가 아니면 페널티
A_bad_EOFF = [] # A가 E인데 다음날 OFF가 아니면 페널티

for d in range(NUM_DAYS - 1):
    # bad_DE = 1 if (A_d==1 and A_{d+1}_E==0)
    bad1 = model.NewBoolVar(f"A_bad_DE_{d}")
    model.Add(x[idxA, d, SHIFT_D] - x[idxA, d+1, SHIFT_E] <= 0).OnlyEnforceIf(bad1.Not())
    # 위 한 줄만으로는 정확한 정의가 안되므로, 표준 방식으로 정의:
    # bad1 >= A_d - A_nextE
    model.Add(bad1 >= x[idxA, d, SHIFT_D] - x[idxA, d+1, SHIFT_E])
    # bad1 <= A_d
    model.Add(bad1 <= x[idxA, d, SHIFT_D])
    A_bad_DE.append(bad1)

    # bad_EOFF = 1 if (A_e==1 and A_{d+1}_OFF==0)
    bad2 = model.NewBoolVar(f"A_bad_EOFF_{d}")
    model.Add(bad2 >= x[idxA, d, SHIFT_E] - x[idxA, d+1, SHIFT_OFF])
    model.Add(bad2 <= x[idxA, d, SHIFT_E])
    A_bad_EOFF.append(bad2)

# =========================================================
# 9) 기타 소프트(기존)
# =========================================================
# 휴무 3연속 지양(비A)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# 일요일 D 1회 이상(비A, 소프트)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# 주말/휴일 편차(비A, 소프트)
special = set(HOLIDAY_INDEXES)
for d in days:
    if is_weekend(START_DATE + timedelta(days=d)):
        special.add(d)
special = sorted(list(special))

special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special
    ))

maxSP = model.NewIntVar(0, len(special), "maxSP")
minSP = model.NewIntVar(0, len(special), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 10) 목적함수(페널티)
# =========================================================
W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500

W_DE_GAP = 200
W_FAIR_D = 150
W_FAIR_E = 150
W_SPECIAL_FAIR = 15

# ✅ 오프 11 목표를 강하게(전원)
W_OFF_TARGET = 300

# ✅ 김영철 패턴 선호 강하게
W_A_DE = 250
W_A_EOFF = 250

model.Minimize(
    W_OFF_TARGET * sum(OFF_dev.values())
    + W_A_DE * sum(A_bad_DE)
    + W_A_EOFF * sum(A_bad_EOFF)
    + W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 11) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 12) 출력: 이름(왼쪽 행) / 날짜(오른쪽 열)
# =========================================================
date_headers = []
for d in range(NUM_DAYS):
    dt = START_DATE + timedelta(days=d)
    date_headers.append(f"{dt.day:02d}({weekday_kor(dt)})")

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "OFF편차"]))

def get_row_shifts(i: int):
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row_shifts = get_row_shifts(i)

    Dn = row_shifts.count("D")
    En = row_shifts.count("E")
    Nn = row_shifts.count("N")
    Wk = Dn + En + Nn
    Of = row_shifts.count("-")
    off_dev_val = abs(Of - OFF_target)

    print("\t".join([name] + row_shifts + [str(Dn), str(En), str(Nn), str(Wk), str(Of), str(off_dev_val)]))

print("\n== 김영철(A) 패턴 체크 ==")
A_row = get_row_shifts(idxA)
bad_de = 0
bad_eoff = 0
for d in range(NUM_DAYS - 1):
    if A_row[d] == "D" and A_row[d+1] != "E":
        bad_de += 1
    if A_row[d] == "E" and A_row[d+1] != "-":
        bad_eoff += 1
print("김영철 D->E 위반:", bad_de)
print("김영철 E->OFF 위반:", bad_eoff)
print("김영철 OFF 개수:", A_row.count("-"), "(목표 11)")


== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	D	E	N	근무	휴무	OFF편차
김영철	-	D	E	-	D	E	-	-	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	8	9	0	17	11	0
홍진우	-	N	-	N	N	-	-	D	N	-	E	N	-	E	E	N	-	E	-	E	E	-	D	D	-	D	D	D	6	6	6	18	10	1
김다영	-	-	D	E	-	D	D	D	D	-	E	E	N	-	-	D	E	N	N	-	-	E	N	N	-	N	-	E	6	6	6	18	10	1
강승민	-	N	N	-	E	E	E	-	D	N	N	-	E	-	-	D	D	D	E	-	D	D	N	-	N	-	-	E	6	6	6	18	10	1
문승환	D	E	-	D	E	-	D	E	-	D	-	N	N	-	-	E	N	N	-	N	N	-	-	D	D	E	E	-	6	6	6	18	10	1
라영일	E	-	-	D	D	-	N	-	E	E	-	D	D	D	D	-	N	-	N	N	-	-	E	-	E	E	N	N	6	6	6	18	10	1
이병욱	N	-	E	N	-	N	-	E	-	D	D	D	D	-	D	N	-	-	E	-	D	N	-	E	E	N	-	-	6	5	6	17	11	0
김동명	D	D	D	E	-	D	E	-	N	N	-	-	E	E	N	-	-	D	D	E	-	E	-	N	N	-	N	-	6	6	6	18	10	1
김선우	E	E	N	-	N	N	-	N	-	E	N	-	-	N	-	E	E	-	D	D	-	D	E	-	D	-	D	D	6	6	6	18	10	1

== 김영철(A) 패턴 체크 ==
김영철 D->E 위반: 0
김영철 E->OFF 위반: 0
김영철 OFF 개수: 11 (목표 11)


In [ ]:
# pip install ortools
from ortools.sat.python import cp_model
from datetime import date, timedelta

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "김영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 3  # 3월

REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# 비워두면 기본 생성 후, 총합 48로 자동 보정
N_REQ_TOTAL = []

HOLIDAY_INDEXES = []

# =========================================================
# 2) 달력 유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    if month == 12:
        nxt = date(year + 1, 1, 1)
    else:
        nxt = date(year, month + 1, 1)
    cur = date(year, month, 1)
    return (nxt - cur).days

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)
days = range(NUM_DAYS)

# =========================================================
# 2-1) N_REQ_TOTAL 기본 생성 + 총합 보정(핵심)
#    - A는 N 금지
#    - 비A는 각자 N=6회(하드) => 총 N 공급량 = 8*6 = 48
#    - 따라서 sum(N_REQ_TOTAL)=48이어야 해가 존재 가능
# =========================================================
TARGET_TOTAL_N = 8 * 6  # 48

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)  # 기본: 평일2 / 주말1
    N_REQ_TOTAL = tmp

cur_total_n = sum(N_REQ_TOTAL)
if cur_total_n > TARGET_TOTAL_N:
    candidates = [
        d for d in range(NUM_DAYS)
        if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 2
    ]
    need = cur_total_n - TARGET_TOTAL_N
    if need > len(candidates):
        raise ValueError("N 총량을 48로 낮출 후보(평일 2->1)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 1

elif cur_total_n < TARGET_TOTAL_N:
    candidates = [
        d for d in range(NUM_DAYS)
        if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 1
    ]
    need = TARGET_TOTAL_N - cur_total_n
    if need > len(candidates):
        raise ValueError("N 총량을 48로 올릴 후보(평일 1->2)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 2

assert len(N_REQ_TOTAL) == NUM_DAYS
assert all(n in [1, 2] for n in N_REQ_TOTAL)
assert sum(N_REQ_TOTAL) == TARGET_TOTAL_N, (sum(N_REQ_TOTAL), TARGET_TOTAL_N)

print("== N 총량 체크 ==")
print("sum(N_REQ_TOTAL) =", sum(N_REQ_TOTAL), "(목표 48)")
print("N(1) 일수:", sum(1 for v in N_REQ_TOTAL if v == 1), " / N(2) 일수:", sum(1 for v in N_REQ_TOTAL if v == 2))

# =========================================================
# 3) 모델 구성
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)
idxA = 0
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
people_nonA = range(1, P_ALL)   # 1~8
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# A(김영철): N 금지
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

# 일일 인원(전체 기준)
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 4) 하드 규칙(비A 핵심)
# =========================================================
# 비A: N=6 (하드)
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# 비A: N 최대 2연속
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

# 비A: N 다음날 D/E 금지(OFF 또는 N만)
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

# 비A: N 2일 후 D 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

# 전원: E 다음날 D 금지
for i in people_all:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

# B/C 특별조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 5) 연속근무 제한(전원): 6연속 근무 금지 / 5연속 지양(소프트)
# =========================================================
work = {(i, d): model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}") for i in people_all for d in days}
for i in people_all:
    for d in days:
        model.Add(work[i, d] == 1 - x[i, d, SHIFT_OFF])

# 전원: 6연속 근무 금지 (=최대 5연속)
for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i, d+k] for k in range(6)) <= 5)

# 5연속 근무 지양(소프트)
five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i, d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i, d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# =========================================================
# 6) 오프 11개 "목표(소프트)" + 근무일수(Work days) 변수
# =========================================================
OFF_target = 11

OFF_cnt = {}
OFF_dev = {}

WORK_cnt = {}
for i in people_all:
    OFF_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{ALL_IDS[i]}")
    model.Add(OFF_cnt[i] == sum(x[i, d, SHIFT_OFF] for d in days))

    OFF_dev[i] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{ALL_IDS[i]}")
    off_diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{ALL_IDS[i]}")
    model.Add(off_diff == OFF_cnt[i] - OFF_target)
    model.AddAbsEquality(OFF_dev[i], off_diff)

    WORK_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"WORKcnt_{ALL_IDS[i]}")
    model.Add(WORK_cnt[i] == sum(work[i, d] for d in days))

# ✅ 핵심 추가 1) 하드: 김영철 근무일수는 다른 누구보다 적으면 안 됨
for i in people_nonA:
    model.Add(WORK_cnt[idxA] >= WORK_cnt[i])

# ✅ 핵심 추가 2) 소프트: 김영철 근무일수는 다른 사람들과 최대한 같게
A_work_gap = {}
for i in people_nonA:
    A_work_gap[i] = model.NewIntVar(0, NUM_DAYS, f"AworkGap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"AworkDiff_{ALL_IDS[i]}")
    model.Add(diff == WORK_cnt[idxA] - WORK_cnt[i])
    model.AddAbsEquality(A_work_gap[i], diff)

# 전체 근무일수 편차도 최소화(소프트)
maxWork = model.NewIntVar(0, NUM_DAYS, "maxWork")
minWork = model.NewIntVar(0, NUM_DAYS, "minWork")
model.AddMaxEquality(maxWork, list(WORK_cnt.values()))
model.AddMinEquality(minWork, list(WORK_cnt.values()))

# =========================================================
# 7) D/E 균일화(비A 소프트 강화)
# =========================================================
D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# =========================================================
# 8) 김영철(A) 패턴 선호(소프트): D→E 선호, E→OFF 선호
# =========================================================
A_bad_DE = []
A_bad_EOFF = []
for d in range(NUM_DAYS - 1):
    bad1 = model.NewBoolVar(f"A_bad_DE_{d}")
    model.Add(bad1 >= x[idxA, d, SHIFT_D] - x[idxA, d+1, SHIFT_E])
    model.Add(bad1 <= x[idxA, d, SHIFT_D])
    A_bad_DE.append(bad1)

    bad2 = model.NewBoolVar(f"A_bad_EOFF_{d}")
    model.Add(bad2 >= x[idxA, d, SHIFT_E] - x[idxA, d+1, SHIFT_OFF])
    model.Add(bad2 <= x[idxA, d, SHIFT_E])
    A_bad_EOFF.append(bad2)

# =========================================================
# 9) 기타 소프트
# =========================================================
# 휴무 3연속 지양(비A)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# 일요일 D 1회 이상(비A, 소프트)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# 주말/휴일 편차(비A, 소프트)
special = set(HOLIDAY_INDEXES)
for d in days:
    if is_weekend(START_DATE + timedelta(days=d)):
        special.add(d)
special = sorted(list(special))

special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special
    ))

maxSP = model.NewIntVar(0, len(special), "maxSP")
minSP = model.NewIntVar(0, len(special), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 10) 목적함수(페널티)
# =========================================================
W_OFF_TARGET = 300
W_A_DE = 250
W_A_EOFF = 250

W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500

W_DE_GAP = 200
W_FAIR_D = 150
W_FAIR_E = 150
W_SPECIAL_FAIR = 15

# ✅ 추가: 김영철 근무일수 "같게" 강제 수준으로 유도
W_A_WORK_EQUAL = 800       # |WORK(A)-WORK(i)| 강하게
W_WORK_FAIR = 200          # 전체 maxWork-minWork 줄이기

model.Minimize(
    W_OFF_TARGET * sum(OFF_dev.values())
    + W_A_WORK_EQUAL * sum(A_work_gap.values())
    + W_WORK_FAIR * (maxWork - minWork)
    + W_A_DE * sum(A_bad_DE)
    + W_A_EOFF * sum(A_bad_EOFF)
    + W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 11) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 12) 출력: 이름(왼쪽 행) / 날짜(오른쪽 열)
# =========================================================
date_headers = []
for d in range(NUM_DAYS):
    dt = START_DATE + timedelta(days=d)
    date_headers.append(f"{dt.day:02d}({weekday_kor(dt)})")

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "OFF편차", "근무일수"]))

def get_row_shifts(i: int):
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

workdays_by_name = {}

for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row_shifts = get_row_shifts(i)

    Dn = row_shifts.count("D")
    En = row_shifts.count("E")
    Nn = row_shifts.count("N")
    Wk = Dn + En + Nn
    Of = row_shifts.count("-")
    off_dev_val = abs(Of - OFF_target)

    workdays_by_name[name] = Wk

    print("\t".join([name] + row_shifts + [str(Dn), str(En), str(Nn), str(Wk), str(Of), str(off_dev_val), str(Wk)]))

print("\n== 김영철 근무일수 체크(하드: 김영철이 더 적으면 안 됨) ==")
a_work = workdays_by_name["김영철"]
print("김영철 근무일수:", a_work)
for nm, wk in workdays_by_name.items():
    if nm == "김영철":
        continue
    if a_work < wk:
        print("❌ 위반:", nm, "근무일수", wk, ">", "김영철", a_work)
        break
else:
    print("✅ 김영철 근무일수가 다른 사람보다 적지 않음(OK)")


== N 총량 체크 ==
sum(N_REQ_TOTAL) = 48 (목표 48)
N(1) 일수: 14  / N(2) 일수: 17

== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	29(일)	30(월)	31(화)	D	E	N	근무	휴무	OFF편차	근무일수
김영철	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	-	D	E	-	D	E	-	D	E	10	10	0	20	11	0	20
홍진우	E	-	-	D	E	-	D	E	N	N	-	-	N	-	-	D	N	N	-	-	D	-	N	-	-	D	E	-	D	E	E	6	6	6	18	13	2	18
김다영	-	D	-	E	E	-	-	D	-	D	D	D	-	D	D	-	E	-	N	N	-	-	E	E	N	N	-	E	E	N	N	7	7	6	20	11	0	20
강승민	E	N	N	-	N	-	-	E	-	D	E	E	E	-	D	E	N	N	-	E	-	N	-	-	D	D	D	D	-	-	D	7	7	6	20	11	0	20
문승환	D	-	D	N	-	-	E	-	E	-	N	N	-	E	-	D	D	D	N	-	-	E	N	N	-	E	-	D	-	E	-	6	6	6	18	13	2	18
라영일	-	-	D	-	-	E	N	-	N	-	-	N	N	-	E	-	E	-	D	D	E	E	-	N	N	-	E	-	D	D	D	6	6	6	18	13	2	18
이병욱	N	-	-	E	-	N	-	-	D	N	N	-	E	E	-	E	-	D	D	-	D	D	D	E	-	N	N	-	E	-	-	6	6	6	18	13	2	18
김동명	D	E	E	-	D	D	D	-	D	E	E	-	D	-	N	N	-	-	E	N	N	-	E	-	D	E	-	-	N	-	N	7	7	6	20	11	0	20
김선우	-	E	-	D	-	D	E	N	-	E

## 3월 수정

In [ ]:
# =========================================================
# 설치(처음 1번)
#   pip install ortools holidayskr
# =========================================================
from ortools.sat.python import cp_model
from datetime import date, timedelta
import calendar
from holidayskr import year_holidays


# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "최영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명(규칙 대상)
ALL_IDS = [A_ID] + STAFF_IDS


# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 3

# 전체 기준 일일 필요 인원
REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# 전체 기준 N 필요 인원(1~2). 비워두면 기본 생성(평일2/주말1) 후 합계 48로 자동 보정
N_REQ_TOTAL = []

# (선택) 특정 날짜를 빨간날로 추가 취급
EXTRA_RED_DATES = [
    # date(2026, 3, 15),
]


# =========================================================
# 2) 달력/유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)  # 토/일

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)
days = range(NUM_DAYS)


# =========================================================
# 2-1) 빨간날(주말 + 공휴일/대체공휴일) -> MIN_OFF
# =========================================================
def get_kr_holidays_in_month(year: int, month: int) -> dict[date, str]:
    out: dict[date, str] = {}
    for dt, name in year_holidays(str(year)):  # dt: datetime.date
        if dt.year == year and dt.month == month:
            out[dt] = name
    return out

def calc_red_dates(year: int, month: int, extra_red_dates: list[date]) -> tuple[set[date], dict[date, str]]:
    last = calendar.monthrange(year, month)[1]
    weekend_set = {date(year, month, d) for d in range(1, last + 1) if is_weekend(date(year, month, d))}
    holiday_name_map = get_kr_holidays_in_month(year, month)
    holiday_set = set(holiday_name_map.keys())
    extra_set = {d for d in extra_red_dates if d.year == year and d.month == month}
    red_dates = weekend_set | holiday_set | extra_set
    return red_dates, holiday_name_map

RED_DATES, HOLIDAY_NAME_MAP = calc_red_dates(YEAR, MONTH, EXTRA_RED_DATES)
MIN_OFF = len(RED_DATES)

print(f"== 빨간날(주말+공휴일+대체공휴일) 최소 휴무 기준 ==\nMIN_OFF = {MIN_OFF}일")
print("== 빨간날 목록 ==")
for dt in sorted(RED_DATES):
    tag = HOLIDAY_NAME_MAP.get(dt, "주말/추가")
    print(dt.isoformat(), tag)


# =========================================================
# 2-2) A(최영철) 고정 패턴 먼저 생성: E -> OFF -> D 반복 (3/1=E)
# =========================================================
def build_A_fixed_E_OFF_D(num_days: int) -> list[str]:
    pat = ["E", "-", "D"]  # 이브닝, 오프, 데이
    return [pat[d % 3] for d in range(num_days)]

A_FIXED = build_A_fixed_E_OFF_D(NUM_DAYS)

# A 고정표가 MIN_OFF 하드(OFF >= MIN_OFF)를 만족 못하면, 애초에 불가능
A_off = A_FIXED.count("-")
if A_off < MIN_OFF:
    raise SystemExit(
        f"[불가능] A(최영철) 고정 OFF={A_off}일 < MIN_OFF={MIN_OFF}일 입니다.\n"
        f"-> A 패턴(이브-오프-데이) 고정에서는 OFF 개수가 고정이라 MIN_OFF를 만족할 수 없어요."
    )

print("\n== A(최영철) 고정 패턴 확인 ==")
print("A_FIXED(앞 15일):", A_FIXED[:15], "| OFF:", A_off, "일")


# =========================================================
# 2-3) N_REQ_TOTAL 기본 생성 + 합계 48로 보정(비A 8명 * 6회)
#   - A는 N을 안 들어가므로, 전체 N 총량은 비A가 전부 채워야 함
# =========================================================
TARGET_TOTAL_N = 8 * 6  # 48

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)  # 기본: 평일2 / 주말1
    N_REQ_TOTAL = tmp

cur_total_n = sum(N_REQ_TOTAL)

if cur_total_n > TARGET_TOTAL_N:
    # 평일 2 -> 1로 낮추기
    candidates = [
        d for d in range(NUM_DAYS)
        if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 2
    ]
    need = cur_total_n - TARGET_TOTAL_N
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 낮출 후보(평일 2->1)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 1

elif cur_total_n < TARGET_TOTAL_N:
    # 평일 1 -> 2로 올리기
    candidates = [
        d for d in range(NUM_DAYS)
        if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 1
    ]
    need = TARGET_TOTAL_N - cur_total_n
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 올릴 후보(평일 1->2)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 2

assert sum(N_REQ_TOTAL) == TARGET_TOTAL_N, (sum(N_REQ_TOTAL), TARGET_TOTAL_N)

print("\n== N 총량 체크 ==")
print("sum(N_REQ_TOTAL) =", sum(N_REQ_TOTAL), "(목표 48)")
print("N(1) 일수:", sum(1 for v in N_REQ_TOTAL if v == 1), "/ N(2) 일수:", sum(1 for v in N_REQ_TOTAL if v == 2))


# =========================================================
# 3) 비A가 채워야 하는 일일 필요 인원 계산(= 전체 - A 고정)
# =========================================================
def req_nonA_for_day(d: int):
    a = A_FIXED[d]
    reqD = REQ_D_TOTAL - (1 if a == "D" else 0)
    reqE = REQ_E_TOTAL - (1 if a == "E" else 0)
    reqN = N_REQ_TOTAL[d]  # A는 N이 없으니 그대로
    if reqD < 0 or reqE < 0:
        raise SystemExit(f"[불가능] {d+1}일에 A가 D/E를 들어가면서 전체 필요인원이 2보다 작게 설정됨.")
    return reqD, reqE, reqN

totalD_nonA = sum(req_nonA_for_day(d)[0] for d in days)
totalE_nonA = sum(req_nonA_for_day(d)[1] for d in days)
totalN_nonA = sum(req_nonA_for_day(d)[2] for d in days)

print("\n== 비A 총 필요 슬롯(검증) ==")
print("D/E/N =", totalD_nonA, totalE_nonA, totalN_nonA, "(비A N 공급량=48)")


# =========================================================
# 4) CP-SAT 모델(비A만 변수로 만들고, A는 고정으로 출력에만 합침)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P = len(STAFF_IDS)  # 8
people = range(P)

idxB = STAFF_IDS.index("B")
idxC = STAFF_IDS.index("C")

shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

# x[p,d,s] : 비A만
x = {(p, d, s): model.NewBoolVar(f"x_{STAFF_IDS[p]}_{d}_{s}")
     for p in people for d in days for s in shifts}

# 하루 1인 1개
for p in people:
    for d in days:
        model.Add(sum(x[p, d, s] for s in shifts) == 1)

# 일일 인원 충족(비A 기준)
for d in days:
    reqD, reqE, reqN = req_nonA_for_day(d)
    model.Add(sum(x[p, d, SHIFT_D] for p in people) == reqD)
    model.Add(sum(x[p, d, SHIFT_E] for p in people) == reqE)
    model.Add(sum(x[p, d, SHIFT_N] for p in people) == reqN)


# =========================================================
# 5) 하드 규칙(비A)
# =========================================================
# N 정확히 6회
for p in people:
    model.Add(sum(x[p, d, SHIFT_N] for d in days) == 6)

# N 최대 2연속
for p in people:
    for d in range(NUM_DAYS - 2):
        model.Add(x[p, d, SHIFT_N] + x[p, d+1, SHIFT_N] + x[p, d+2, SHIFT_N] <= 2)

# N 다음날 D/E 금지
for p in people:
    for d in range(NUM_DAYS - 1):
        model.Add(x[p, d, SHIFT_N] + x[p, d+1, SHIFT_D] <= 1)
        model.Add(x[p, d, SHIFT_N] + x[p, d+1, SHIFT_E] <= 1)

# N 2일 후 D 금지
for p in people:
    for d in range(NUM_DAYS - 2):
        model.Add(x[p, d, SHIFT_N] + x[p, d+2, SHIFT_D] <= 1)

# (비A만) E 다음날 D 금지
for p in people:
    for d in range(NUM_DAYS - 1):
        model.Add(x[p, d, SHIFT_E] + x[p, d+1, SHIFT_D] <= 1)

# B/C 특별 조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)


# =========================================================
# 6) 소프트 규칙(가능하면 맞추기)
# =========================================================
# 카운트
D_cnt, E_cnt, OFF_cnt, WORK_cnt = {}, {}, {}, {}
for p in people:
    D_cnt[p] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{STAFF_IDS[p]}")
    E_cnt[p] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{STAFF_IDS[p]}")
    OFF_cnt[p] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{STAFF_IDS[p]}")
    WORK_cnt[p] = model.NewIntVar(0, NUM_DAYS, f"WORKcnt_{STAFF_IDS[p]}")

    model.Add(D_cnt[p] == sum(x[p, d, SHIFT_D] for d in days))
    model.Add(E_cnt[p] == sum(x[p, d, SHIFT_E] for d in days))
    model.Add(OFF_cnt[p] == sum(x[p, d, SHIFT_OFF] for d in days))
    model.Add(WORK_cnt[p] == sum(x[p, d, SHIFT_D] + x[p, d, SHIFT_E] + x[p, d, SHIFT_N] for d in days))

# (하드) 휴무 최소: MIN_OFF 이상
for p in people:
    model.Add(OFF_cnt[p] >= MIN_OFF)

# (소프트) 휴무 11일 목표
OFF_target = 11
OFF_dev = {}
for p in people:
    OFF_dev[p] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{STAFF_IDS[p]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{STAFF_IDS[p]}")
    model.Add(diff == OFF_cnt[p] - OFF_target)
    model.AddAbsEquality(OFF_dev[p], diff)

# (소프트) 개인 D=E 가깝게
DE_gap = {}
for p in people:
    DE_gap[p] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{STAFF_IDS[p]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{STAFF_IDS[p]}")
    model.Add(diff == D_cnt[p] - E_cnt[p])
    model.AddAbsEquality(DE_gap[p], diff)

# (소프트) D/E 편차 최소화
maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# (하드) 6연속 근무 금지(=최대 5연속)
work = {(p, d): model.NewBoolVar(f"work_{STAFF_IDS[p]}_{d}") for p in people for d in days}
for p in people:
    for d in days:
        model.Add(work[p, d] == 1 - x[p, d, SHIFT_OFF])
for p in people:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[p, d+k] for k in range(6)) <= 5)

# (소프트) 5연속 근무 지양
five_consec_flags = []
for p in people:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{STAFF_IDS[p]}_{d}")
        model.Add(sum(work[p, d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[p, d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# (소프트) 휴무 3연속 지양
triple_off_flags = []
for p in people:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{STAFF_IDS[p]}_{d}")
        model.Add(x[p, d, SHIFT_OFF] + x[p, d+1, SHIFT_OFF] + x[p, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[p, d, SHIFT_OFF] + x[p, d+1, SHIFT_OFF] + x[p, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# (소프트) 일요일 D 월 1회 이상
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for p in people:
    m = model.NewBoolVar(f"missSunD_{STAFF_IDS[p]}")
    sum_sunD = sum(x[p, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[p] = m

# (소프트) 빨간날 근무 편중 최소화
special_days = sorted([(dt - START_DATE).days for dt in RED_DATES])  # 0-based index list
special_work = {}
for p in people:
    special_work[p] = model.NewIntVar(0, len(special_days), f"SPW_{STAFF_IDS[p]}")
    model.Add(special_work[p] == sum(
        x[p, d, SHIFT_D] + x[p, d, SHIFT_E] + x[p, d, SHIFT_N]
        for d in special_days
    ))
maxSP = model.NewIntVar(0, len(special_days), "maxSP")
minSP = model.NewIntVar(0, len(special_days), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))


# =========================================================
# 7) 목적함수(페널티)
# =========================================================
W_OFF_TARGET = 250
W_DE_GAP = 200
W_FAIR_D = 120
W_FAIR_E = 120
W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500
W_SPECIAL_FAIR = 20

model.Minimize(
    W_OFF_TARGET * sum(OFF_dev.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_SPECIAL_FAIR * (maxSP - minSP)
)


# =========================================================
# 8) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("\n해를 찾지 못했습니다.")
    print("가능 원인: A 고정 패턴 + (비A N=6 + B/C 제약 + 일일 인원) 조합이 충돌")
    raise SystemExit


# =========================================================
# 9) 출력(이름 행 / 날짜 열)  + 휴무 달력
# =========================================================
def get_nonA_row_shifts(p: int) -> list[str]:
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[p, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

def print_off_calendar(year: int, month: int, shifts_list: list[str], title: str, off_symbol: str = "-", week_start: int = 0):
    cal = calendar.Calendar(firstweekday=week_start)
    print(f"\n[{title}] {year}-{month:02d} 휴무 달력 (휴무: [dd])")
    print("Mo Tu We Th Fr Sa Su" if week_start == 0 else "Su Mo Tu We Th Fr Sa")
    for week in cal.monthdayscalendar(year, month):
        cells = []
        for d in week:
            if d == 0:
                cells.append("  ")
            else:
                is_off = (shifts_list[d - 1] == off_symbol)
                cells.append(f"[{d:02d}]" if is_off else f" {d:02d}")
        print(" ".join(cells))

date_headers = [f"{(START_DATE + timedelta(days=d)).day:02d}({weekday_kor(START_DATE + timedelta(days=d))})"
                for d in range(NUM_DAYS)]

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "MIN_OFF"]))

# A 먼저 출력(고정)
A_row = A_FIXED[:]  # E / - / D 반복
A_D = A_row.count("D")
A_E = A_row.count("E")
A_N = A_row.count("N")
A_W = A_D + A_E + A_N
A_O = A_row.count("-")
print("\t".join([NAME_MAP["A"]] + A_row + [str(A_D), str(A_E), str(A_N), str(A_W), str(A_O), str(MIN_OFF)]))

# 비A 출력(해결값)
rows_by_name = {NAME_MAP["A"]: A_row}
for p, sid in enumerate(STAFF_IDS):
    name = NAME_MAP[sid]
    row = get_nonA_row_shifts(p)
    rows_by_name[name] = row

    Dn = row.count("D")
    En = row.count("E")
    Nn = row.count("N")
    Wk = Dn + En + Nn
    Of = row.count("-")

    print("\t".join([name] + row + [str(Dn), str(En), str(Nn), str(Wk), str(Of), str(MIN_OFF)]))

print("\n== 휴무 달력 출력 ==")
for name, row in rows_by_name.items():
    print_off_calendar(YEAR, MONTH, row, title=name, week_start=0)

print("\n== 날짜별 인원 체크(전체 기준) ==")
print("\t".join(["항목"] + date_headers))
D_line = ["D(전체)"]
E_line = ["E(전체)"]
N_line = ["N(전체)"]
OFF_line = ["-(전체)"]
for d in range(NUM_DAYS):
    # A는 고정값
    aD = 1 if A_FIXED[d] == "D" else 0
    aE = 1 if A_FIXED[d] == "E" else 0
    aN = 0
    aO = 1 if A_FIXED[d] == "-" else 0

    d_cnt = aD + sum(solver.Value(x[p, d, SHIFT_D]) for p in people)
    e_cnt = aE + sum(solver.Value(x[p, d, SHIFT_E]) for p in people)
    n_cnt = aN + sum(solver.Value(x[p, d, SHIFT_N]) for p in people)
    off_cnt = aO + sum(solver.Value(x[p, d, SHIFT_OFF]) for p in people)

    D_line.append(str(d_cnt))
    E_line.append(str(e_cnt))
    N_line.append(str(n_cnt))
    OFF_line.append(str(off_cnt))

print("\t".join(D_line))
print("\t".join(E_line))
print("\t".join(N_line))
print("\t".join(OFF_line))

== 빨간날(주말+공휴일+대체공휴일) 최소 휴무 기준 ==
MIN_OFF = 10일
== 빨간날 목록 ==
2026-03-01 3·1절
2026-03-02 대체 공휴일(3·1절)
2026-03-07 주말/추가
2026-03-08 주말/추가
2026-03-14 주말/추가
2026-03-15 주말/추가
2026-03-21 주말/추가
2026-03-22 주말/추가
2026-03-28 주말/추가
2026-03-29 주말/추가

== A(최영철) 고정 패턴 확인 ==
A_FIXED(앞 15일): ['E', '-', 'D', 'E', '-', 'D', 'E', '-', 'D', 'E', '-', 'D', 'E', '-', 'D'] | OFF: 10 일

== N 총량 체크 ==
sum(N_REQ_TOTAL) = 48 (목표 48)
N(1) 일수: 14 / N(2) 일수: 17

== 비A 총 필요 슬롯(검증) ==
D/E/N = 52 51 48 (비A N 공급량=48)

== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	29(일)	30(월)	31(화)	D	E	N	근무	휴무	MIN_OFF
최영철	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	10	11	0	21	10	10
홍진우	-	-	D	-	D	-	D	D	N	N	-	E	E	E	E	-	D	E	N	N	-	-	D	E	N	-	N	-	E	-	D	7	7	6	20	11	10
김다영	-	-	E	-	D	-	-	E	-	D	E	-	D	D	-	N	N	-	E	-	D	D	N	N	-	N	-	-	E	E	N	6	6	6	18	13	10
강승민	D	E	-	D	-	E	-	-	N	-	N	N	-	E